# Heat Stress Alerting Simulation — Mohali District, Punjab, India (2022)

For full methodology and data provenance, see the accompanying paper and the GitHub repository.

## 1. Install dependencies

In [2]:
!pip install cfgrib xarray eccodes --quiet


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.1/49.1 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.6/91.6 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 59.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 77.2 MB/s eta 0:00:00


In [3]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path

warnings.filterwarnings("ignore")


## 2. Constants

In [4]:
LAT_MIN, LAT_MAX = 30.60, 30.85
LON_MIN, LON_MAX = 76.65, 76.95

N_NODES = 25

UHI_MIN, UHI_MAX = 0.0, 2.5

FORECAST_DEPARTURE_THRESHOLD = 4.5

UPSDMA_TA_THRESHOLDS = {
    "Yellow": 37.62,
    "Orange": 40.39,
    "Red":    43.83,
}

TIER_PERCENTILES = {"Yellow": 75, "Orange": 85, "Red": 95}

SEVERITY_RANK = {"None": 0, "Yellow": 1, "Orange": 2, "Red": 3}
RANK_TO_SEVERITY = {v: k for k, v in SEVERITY_RANK.items()}

TIER_COLOURS = {
    "None":   "#d9d9d9",
    "Yellow": "#f1c400",
    "Orange": "#e67e22",
    "Red":    "#c0392b",
}
TIER_PLOT_ORDER = ["None", "Yellow", "Orange", "Red"]

SEGMENTS = {
    "Elderly (60+)":          0.115,
    "Outdoor Workers":        0.392,
    "Children (<15)":         0.192,
    "General Population":     0.301,
}
TOTAL_POP = 10_000
APP_REGISTRATION_RATE = 0.45

MC_ITERATIONS = 1_000

NETWORK_CONDITIONS = {
    "Stable 4G": (0.05,  1.5,  0.5),
    "Degraded 2G": (0.30, 10.0, 5.0),
}
SMS_PARAMS = (0.00, 60.0, 10.0)

CHANNEL_RELIABILITY = {
    "App Push":   0.97,
    "News Feed":  0.99,
    "Radio Feed": 0.99,
    "SMS":        0.98,
}

SIM_START = "2022-01-01"
SIM_END   = "2022-12-31"

HEAT_SEASON_START = "2022-01-01"
HEAT_SEASON_END   = "2022-09-30"


## 3. Data loading

In [5]:
def load_era5_grib(grib_path: str, apply_sim_slice: bool = True) -> pd.DataFrame:
    try:
        import cfgrib
        import xarray as xr
    except ImportError:
        raise ImportError("cfgrib and xarray required. Run install cell first.")

    print(f"  Loading GRIB: {grib_path}")

    try:
        ds = xr.open_dataset(grib_path, engine="cfgrib",
                             backend_kwargs={'filter_by_keys': {'stepType': 'instant'}})
        print(f"  Variables in file (primary load): {list(ds.data_vars)}")
    except Exception as e:
        print(f"  Error opening GRIB: {e}")
        print("  Falling back to synthetic data.")
        return generate_synthetic_era5()

    df = pd.DataFrame()

    for var_name in ["t2m"]:
        if var_name in ds.data_vars:
            try:
                da = ds[var_name]
                if "latitude" in da.dims and "longitude" in da.dims:
                    da = da.sel(
                        latitude=slice(LAT_MAX, LAT_MIN),
                        longitude=slice(LON_MIN, LON_MAX),
                    ).mean(dim=["latitude", "longitude"])
                elif "lat" in da.dims and "lon" in da.dims:
                    da = da.sel(
                        lat=slice(LAT_MIN, LAT_MAX),
                        lon=slice(LON_MIN, LON_MAX),
                    ).mean(dim=["lat", "lon"])

                df["T_a"] = da.values - 273.15
                print(f"  Loaded T_a from '{var_name}'")
                break
            except Exception as e:
                print(f"    Could not load {var_name}: {e}")

    for var_name in ["d2m"]:
        if var_name in ds.data_vars:
            try:
                da = ds[var_name]
                if "latitude" in da.dims and "longitude" in da.dims:
                    da = da.sel(
                        latitude=slice(LAT_MAX, LAT_MIN),
                        longitude=slice(LON_MIN, LON_MAX),
                    ).mean(dim=["latitude", "longitude"])
                elif "lat" in da.dims and "lon" in da.dims:
                    da = da.sel(
                        lat=slice(LAT_MIN, LAT_MAX),
                        lon=slice(LON_MIN, LON_MAX),
                    ).mean(dim=["lat", "lon"])

                df["Td"] = da.values - 273.15
                print(f"  Loaded Td from '{var_name}'")
                break
            except Exception as e:
                print(f"    Could not load {var_name}: {e}")

    ssrd_loaded_successfully = False
    try:
        ds_ssrd = xr.open_dataset(
            grib_path, engine="cfgrib",
            backend_kwargs={"filter_by_keys": {"paramId": 169}},
        )
        da = ds_ssrd["ssrd"]
        if "latitude" in da.dims and "longitude" in da.dims:
            da = da.sel(
                latitude=slice(LAT_MAX, LAT_MIN),
                longitude=slice(LON_MIN, LON_MAX),
            ).mean(dim=["latitude", "longitude"])
        elif "lat" in da.dims and "lon" in da.dims:
            da = da.sel(
                lat=slice(LAT_MIN, LAT_MAX),
                lon=slice(LON_MIN, LON_MAX),
            ).mean(dim=["lat", "lon"])

        ssrd_vals = da.values.copy()
        ssrd_vals = np.diff(ssrd_vals, prepend=ssrd_vals[0])
        ssrd_vals = np.clip(ssrd_vals, 0, None) / 3600.0
        df["SSRD"] = ssrd_vals
        print("  Loaded SSRD from 'ssrd' (paramId=169, filtered open)")
    except Exception as e:
        print(f"  Could not load SSRD via filter_by_keys: {e}")

    if not ssrd_loaded_successfully:
        try:
            ssrd_ds_targeted = xr.open_dataset(grib_path, engine="cfgrib",
                                               backend_kwargs={'filter_by_keys': {'paramId': 169, 'stepType': 'instant'}})
            if "ssrd" in ssrd_ds_targeted.data_vars:
                print(f"  Found 'ssrd' via targeted load (paramId 169).")
                da = ssrd_ds_targeted['ssrd']
                if "latitude" in da.dims and "longitude" in da.dims:
                    da = da.sel(
                        latitude=slice(LAT_MAX, LAT_MIN),
                        longitude=slice(LON_MIN, LON_MAX),
                    ).mean(dim=["latitude", "longitude"])
                elif "lat" in da.dims and "lon" in da.dims:
                    da = da.sel(
                        lat=slice(LAT_MIN, LAT_MAX),
                        lon=slice(LON_MIN, LON_MAX),
                    ).mean(dim=["lat", "lon"])

                ssrd_vals = da.values.copy()
                ssrd_vals = np.diff(ssrd_vals, prepend=ssrd_vals[0])
                ssrd_vals = np.clip(ssrd_vals, 0, None) / 3600.0
                df["SSRD"] = ssrd_vals
                print(f"  Loaded SSRD from targeted load (paramId 169).")
                ssrd_loaded_successfully = True
            else:
                print(f"  Targeted load for paramId 169 did not yield 'ssrd'.")
        except Exception as e:
            print(f"  Error trying to load SSRD separately with paramId 169: {e}")

    if "time" in ds.coords:
        df.index = pd.to_datetime(ds["time"].values)
    else:
        print("  Warning: no time coordinate found, using default range.")
        df.index = pd.date_range(SIM_START, periods=len(df), freq="h")

    df.index.name = "time"

    if "SSRD" not in df.columns:
        print("  Warning: solar radiation not loaded, using default 200 W/m^2")
        df["SSRD"] = 200.0

    if "T_a" not in df.columns or "Td" not in df.columns:
        print("  Error: missing T_a or Td — falling back to synthetic data.")
        return generate_synthetic_era5()
    df = df.dropna()
    df = df.sort_index()
    if apply_sim_slice:
        df = df.loc[SIM_START:SIM_END]

    print(f"  Loaded {len(df):,} records ({df.index[0]} to {df.index[-1]})")
    return df

def load_era5_baseline_grib(jan_jun_path: str, jul_dec_path: str) -> pd.DataFrame:
    df_h1 = load_era5_grib(jan_jun_path, apply_sim_slice=False)
    df_h2 = load_era5_grib(jul_dec_path, apply_sim_slice=False)
    df_baseline = pd.concat([df_h1, df_h2]).sort_index()
    df_baseline = df_baseline[~df_baseline.index.duplicated(keep="first")]
    print(f"  Baseline: {len(df_baseline):,} records "
          f"({df_baseline.index[0]} to {df_baseline.index[-1]})")
    return df_baseline

def generate_synthetic_era5(start: str = None, end: str = None, seed: int = 42) -> pd.DataFrame:
    start = start or SIM_START
    end = end or SIM_END
    rng = np.random.default_rng(seed)
    times = pd.date_range(start, end, freq="h")
    n = len(times)
    doy = times.day_of_year.values
    hour = times.hour.values

    T_seasonal = 18.0 + 24.0 * np.sin(np.pi * (doy - 15) / 180) ** 2
    T_diurnal = 7.0 * np.sin(np.pi * (hour - 4) / 12) * (hour >= 4) * (hour <= 16)
    T_noise = rng.normal(0, 1.2, n)
    T_a = T_seasonal + T_diurnal + T_noise

    Td_offset = -18 + 12 * np.sin(np.pi * (doy - 60) / 200)
    Td = T_a + Td_offset + rng.normal(0, 1.0, n)
    Td = np.minimum(Td, T_a)

    solar_peak = 600 + 200 * np.sin(np.pi * (doy - 60) / 180)
    solar_hour_factor = np.maximum(0, np.sin(np.pi * (hour - 6) / 12))
    SSRD = solar_peak * solar_hour_factor + rng.uniform(0, 30, n)
    SSRD = np.clip(SSRD, 0, None)

    df = pd.DataFrame({"T_a": T_a, "Td": Td, "SSRD": SSRD}, index=times)
    df.index.name = "time"
    print(f"  Generated {len(df):,} synthetic hourly records "
          f"({df.index[0].date()} to {df.index[-1].date()})")
    return df


## 4. Upload ERA5 GRIB files (optional; falls back to synthetic data if skipped)

In [7]:
from google.colab import files

GRIB_PATH = None

try:
    uploaded = files.upload()
    if uploaded:
        GRIB_PATH = list(uploaded.keys())[0]
        print(f"Using uploaded file: {GRIB_PATH}")
        print("Upload Jan-Jun baseline GRIB:")
        uploaded = files.upload()
        BASELINE_JAN_JUN_PATH = list(uploaded.keys())[0] if uploaded else None

        print("Upload Jul-Dec baseline GRIB:")
        uploaded = files.upload()
        BASELINE_JUL_DEC_PATH = list(uploaded.keys())[0] if uploaded else None
    else:
        print("No file selected — will use synthetic data.")
except Exception as e:
    print(f"Upload skipped/failed ({e}) — will use synthetic data.")

OUT_DIR = Path("/content/heat_stress_results")
OUT_DIR.mkdir(parents=True, exist_ok=True)


Saving Mohali2022New.grib to Mohali2022New.grib
Using uploaded file: Mohali2022New.grib
Upload Jan-Jun baseline GRIB:


Saving 19991_to_2020.grib to 19991_to_2020.grib
Upload Jul-Dec baseline GRIB:


Saving 19991_to_2020_two.grib to 19991_to_2020_two.grib


## 5. Yellow / Orange / Red tier thresholds (UPSDMA framework)

In [ ]:
def compute_baseline_tier_thresholds(baseline_idx_df: pd.DataFrame = None) -> dict:
    thresholds = {"T_a": dict(UPSDMA_TA_THRESHOLDS)}

    if baseline_idx_df is not None:
        baseline_idx = baseline_idx_df
    else:
        baseline_raw = generate_synthetic_era5(start="2000-01-01", end="2000-12-31", seed=123)
        baseline_idx = compute_indices(baseline_raw)

    daily_max = baseline_idx.resample("D").max()
    for index_name in ["HI", "ESI"]:
        vals = daily_max[index_name].dropna().values
        thresholds[index_name] = {
            tier: round(float(np.percentile(vals, pct)), 2)
            for tier, pct in TIER_PERCENTILES.items()
        }
    return thresholds


## 6. Heat stress index computation

In [8]:
def relative_humidity_from_dewpoint(T_a: np.ndarray, Td: np.ndarray) -> np.ndarray:
    a, b = 17.625, 243.04
    gamma_T  = a * T_a / (b + T_a)
    gamma_Td = a * Td / (b + Td)
    RH = 100.0 * np.exp(gamma_Td - gamma_T)
    return np.clip(RH, 0.0, 100.0)

def compute_heat_index(T_a: np.ndarray, RH: np.ndarray) -> np.ndarray:
    T_f = T_a * 9.0 / 5.0 + 32.0

    hi_simple = 0.5 * (T_f + 61.0 + (T_f - 68.0) * 1.2 + RH * 0.094)

    hi_full = (
        -42.379
        + 2.04901523 * T_f
        + 10.14333127 * RH
        - 0.22475541 * T_f * RH
        - 6.83783e-3 * T_f**2
        - 5.481717e-2 * RH**2
        + 1.22874e-3 * T_f**2 * RH
        + 8.5282e-4 * T_f * RH**2
        - 1.99e-6 * T_f**2 * RH**2
    )

    adj_low_rh  = ((13.0 - RH) / 4.0) * np.sqrt((17.0 - np.abs(T_f - 95.0)) / 17.0)
    adj_high_rh = (RH - 85.0) / 10.0 * ((87.0 - T_f) / 5.0)

    hi_full = np.where((RH < 13) & (T_f >= 80) & (T_f <= 112), hi_full - adj_low_rh, hi_full)
    hi_full = np.where((RH > 85) & (T_f >= 80) & (T_f <= 87), hi_full + adj_high_rh, hi_full)

    hi_f = np.where(T_a >= 27.0, hi_full, hi_simple)
    hi_c = (hi_f - 32.0) * 5.0 / 9.0
    return hi_c

def compute_esi(T_a: np.ndarray, RH: np.ndarray, SSRD: np.ndarray) -> np.ndarray:
    ESI = (
        0.63 * T_a
        - 0.03 * RH
        + 0.002 * SSRD
        + 0.01 * T_a * RH
        - 0.073 / (0.1 + SSRD)
        - 3.0
    )
    return ESI

def compute_indices(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["RH"]  = relative_humidity_from_dewpoint(df["T_a"].values, df["Td"].values)
    df["HI"]  = compute_heat_index(df["T_a"].values, df["RH"].values)
    df["ESI"] = compute_esi(df["T_a"].values, df["RH"].values, df["SSRD"].values)
    return df


## 7. Sensor node simulation and severity classification

In [9]:
def simulate_nodes(df: pd.DataFrame, seed: int = 0) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    uhi_offsets = rng.uniform(UHI_MIN, UHI_MAX, N_NODES)

    node_hi  = np.zeros((len(df), N_NODES))
    node_esi = np.zeros((len(df), N_NODES))
    node_ta  = np.zeros((len(df), N_NODES))

    for i, offset in enumerate(uhi_offsets):
        T_node = df["T_a"].values + offset
        RH_node = relative_humidity_from_dewpoint(T_node, df["Td"].values)
        node_hi[:, i]  = compute_heat_index(T_node, RH_node)
        node_esi[:, i] = compute_esi(T_node, RH_node, df["SSRD"].values)
        node_ta[:, i]  = T_node

    result = df.copy()
    result["HI_max"]  = node_hi.max(axis=1)
    result["ESI_max"] = node_esi.max(axis=1)
    result["Ta_max"]  = node_ta.max(axis=1)
    return result

def classify_severity(values: np.ndarray, tiers: dict) -> np.ndarray:
    rank = np.zeros(values.shape, dtype=int)
    rank[values >= tiers["Yellow"]] = SEVERITY_RANK["Yellow"]
    rank[values >= tiers["Orange"]] = SEVERITY_RANK["Orange"]
    rank[values >= tiers["Red"]]    = SEVERITY_RANK["Red"]
    return rank

def detect_severity_tiers(df: pd.DataFrame, tier_thresholds: dict) -> pd.DataFrame:
    df = df.copy()
    for index_name, col in [("HI", "HI_max"), ("ESI", "ESI_max"), ("T_a", "Ta_max")]:
        rank = classify_severity(df[col].values, tier_thresholds[index_name])
        df[f"severity_rank_{index_name}"] = rank
        df[f"severity_{index_name}"] = [RANK_TO_SEVERITY[r] for r in rank]
    return df

def daily_peak_severity(df: pd.DataFrame, index_name: str) -> pd.Series:
    rank_col = f"severity_rank_{index_name}"
    daily_rank = df[rank_col].resample("D").max()
    return daily_rank.map(RANK_TO_SEVERITY)


In [10]:
def simulate_nodes(df: pd.DataFrame, seed: int = 0) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    uhi_offsets = rng.uniform(UHI_MIN, UHI_MAX, N_NODES)

    node_hi  = np.zeros((len(df), N_NODES))
    node_esi = np.zeros((len(df), N_NODES))
    node_ta  = np.zeros((len(df), N_NODES))

    for i, offset in enumerate(uhi_offsets):
        T_node = df["T_a"].values + offset
        RH_node = relative_humidity_from_dewpoint(T_node, df["Td"].values)
        node_hi[:, i]  = compute_heat_index(T_node, RH_node)
        node_esi[:, i] = compute_esi(T_node, RH_node, df["SSRD"].values)
        node_ta[:, i]  = T_node

    result = df.copy()
    result["HI_max"]  = node_hi.max(axis=1)
    result["ESI_max"] = node_esi.max(axis=1)
    result["Ta_max"]  = node_ta.max(axis=1)
    return result

def classify_severity(values: np.ndarray, tiers: dict) -> np.ndarray:
    rank = np.zeros(values.shape, dtype=int)
    rank[values >= tiers["Yellow"]] = SEVERITY_RANK["Yellow"]
    rank[values >= tiers["Orange"]] = SEVERITY_RANK["Orange"]
    rank[values >= tiers["Red"]]    = SEVERITY_RANK["Red"]
    return rank

def detect_severity_tiers(df: pd.DataFrame, tier_thresholds: dict) -> pd.DataFrame:
    df = df.copy()
    for index_name, col in [("HI", "HI_max"), ("ESI", "ESI_max"), ("T_a", "Ta_max")]:
        rank = classify_severity(df[col].values, tier_thresholds[index_name])
        df[f"severity_rank_{index_name}"] = rank
        df[f"severity_{index_name}"] = [RANK_TO_SEVERITY[r] for r in rank]
    return df

def daily_peak_severity(df: pd.DataFrame, index_name: str) -> pd.Series:
    rank_col = f"severity_rank_{index_name}"
    daily_rank = df[rank_col].resample("D").max()
    return daily_rank.map(RANK_TO_SEVERITY)


## 8. Dual-mode sensor operation

In [11]:
def compute_climatological_normal(baseline_df: pd.DataFrame) -> pd.Series:
    daily_max = baseline_df["T_a"].resample("D").max()
    doy = daily_max.index.day_of_year
    normal_by_doy = {}
    for d in range(1, 367):
        mask = ((doy >= d - 15) & (doy <= d + 15)) | (doy >= d + 351) | (doy <= d - 351)
        normal_by_doy[d] = daily_max[mask].mean()
    return pd.Series(normal_by_doy)

def compute_forecast_departure(df: pd.DataFrame, clim_mean_series: pd.Series = None) -> pd.Series:
    if clim_mean_series is None:
        doy_vals = np.arange(1, 367)
        clim_mean = 12.0 + 20.0 * np.sin(np.pi * (doy_vals - 15) / 180) ** 2
        clim_mean_series = pd.Series(clim_mean, index=doy_vals)

    daily_max = df["T_a"].resample("D").max()
    doy_daily = daily_max.index.day_of_year
    normal = doy_daily.map(clim_mean_series)
    departure = daily_max.values - normal
    departure_series = pd.Series(departure, index=daily_max.index)
    return departure_series.rolling(7, min_periods=1).max()

def simulate_dual_mode(df: pd.DataFrame, clim_mean_series: pd.Series = None):
    forecast = compute_forecast_departure(df, clim_mean_series=clim_mean_series)
    high_alert_days = forecast[forecast >= FORECAST_DEPARTURE_THRESHOLD].index

    daily_dates = pd.date_range(SIM_START, SIM_END, freq="D")
    mode_daily = pd.Series("low_power", index=daily_dates, name="mode")
    for d in high_alert_days:
        if d in mode_daily.index:
            mode_daily[d] = "high_alert"

    df = df.copy()
    df["mode"] = df.index.normalize().map(mode_daily).fillna("low_power")
    return df, mode_daily, forecast


## 9. Result summaries

In [12]:
def summarise_severity_tiers(df: pd.DataFrame) -> pd.DataFrame:
    total_days = df.resample("D")["T_a"].count().shape[0]
    rows = []
    for index_name in ["HI", "ESI", "T_a"]:
        daily_peak = daily_peak_severity(df, index_name)
        counts = daily_peak.value_counts()
        for tier in ["Red", "Orange", "Yellow", "None"]:
            n = int(counts.get(tier, 0))
            rows.append({
                "Index Pipeline": index_name,
                "Severity Tier (Daily Peak)": tier,
                "Days": n,
                "% of Days": round(100.0 * n / total_days, 1),
            })
    return pd.DataFrame(rows)

def summarise_severity_hours(df: pd.DataFrame) -> pd.DataFrame:
    total_hours = len(df)
    rows = []
    for index_name in ["HI", "ESI", "T_a"]:
        counts = df[f"severity_{index_name}"].value_counts()
        for tier in ["Red", "Orange", "Yellow", "None"]:
            n = int(counts.get(tier, 0))
            rows.append({
                "Index Pipeline": index_name,
                "Severity Tier (Hourly)": tier,
                "Hours": n,
                "% of Hours": round(100.0 * n / total_hours, 1),
            })
    return pd.DataFrame(rows)

def summarise_severity_by_mode(df_mode: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for index_name in ["HI", "ESI", "T_a"]:
        col = f"severity_{index_name}"
        for mode in ["low_power", "high_alert"]:
            subset = df_mode[df_mode["mode"] == mode]
            counts = subset[col].value_counts()
            for tier in ["Red", "Orange", "Yellow"]:
                rows.append({
                    "Index Pipeline": index_name,
                    "Mode": mode,
                    "Severity Tier": tier,
                    "Hours": int(counts.get(tier, 0)),
                })
    return pd.DataFrame(rows)

def summarise_dual_mode(mode_daily: pd.Series, forecast: pd.Series) -> dict:
    n_high = (mode_daily == "high_alert").sum()
    n_low  = (mode_daily == "low_power").sum()
    total  = len(mode_daily)

    is_high = (mode_daily == "high_alert").astype(int)
    transitions = is_high.diff().fillna(0)
    n_activations    = int((transitions == 1).sum())
    n_deactivations  = int((transitions == -1).sum())

    peak_departure     = forecast.max()
    peak_departure_day = forecast.idxmax()

    return {
        "Total Days":                    total,
        "Low-Power Mode Days":           int(n_low),
        "High-Alert Mode Days":          int(n_high),
        "% Days in High-Alert Mode":     round(100.0 * n_high / total, 1),
        "High-Alert Activations":        n_activations,
        "High-Alert Deactivations":      n_deactivations,
        "Peak Forecast Departure (C)":   round(float(peak_departure), 2),
        "Peak Departure Date":           str(peak_departure_day.date()),
    }


## 10. Multi-channel delivery — Monte Carlo simulation

In [14]:
def monte_carlo_delivery(seed: int = 99) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    rows = []

    data_channels = {
        "App Push":   CHANNEL_RELIABILITY["App Push"],
        "News Feed":  CHANNEL_RELIABILITY["News Feed"],
        "Radio Feed": CHANNEL_RELIABILITY["Radio Feed"],
    }

    for net_name, (pkt_loss, lat_mean, lat_std) in NETWORK_CONDITIONS.items():
        for ch_name, base_reliability in data_channels.items():
            success_flags = []
            latencies     = []
            for _ in range(MC_ITERATIONS):
                network_ok = rng.random() > pkt_loss
                infra_ok   = rng.random() < base_reliability
                delivered  = network_ok and infra_ok
                success_flags.append(int(delivered))
                if delivered:
                    latencies.append(max(0, rng.normal(lat_mean, lat_std)))

            success_rate = 100.0 * np.mean(success_flags)
            mean_lat     = np.mean(latencies) if latencies else np.nan
            p95_lat      = np.percentile(latencies, 95) if latencies else np.nan

            rows.append({
                "Channel":              ch_name,
                "Network Condition":    net_name,
                "Delivery Rate (%)":    round(success_rate, 1),
                "Mean Latency (s)":     round(mean_lat, 2) if not np.isnan(mean_lat) else "N/A",
                "P95 Latency (s)":      round(p95_lat, 2)  if not np.isnan(p95_lat)  else "N/A",
            })

    pkt_loss_sms, lat_mean_sms, lat_std_sms = SMS_PARAMS
    sms_reliability = CHANNEL_RELIABILITY["SMS"]
    for net_name in NETWORK_CONDITIONS:
        success_flags = []
        latencies     = []
        for _ in range(MC_ITERATIONS):
            infra_ok  = rng.random() < sms_reliability
            delivered = infra_ok
            success_flags.append(int(delivered))
            if delivered:
                latencies.append(max(0, rng.normal(lat_mean_sms, lat_std_sms)))

        rows.append({
            "Channel":              "SMS",
            "Network Condition":    net_name,
            "Delivery Rate (%)":    round(100.0 * np.mean(success_flags), 1),
            "Mean Latency (s)":     round(np.mean(latencies), 2),
            "P95 Latency (s)":      round(np.percentile(latencies, 95), 2),
        })

    return pd.DataFrame(rows)


## 11. Figures

In [15]:
def plot_severity_tier_comparison(tier_day_summary: pd.DataFrame, out_dir: Path):
    pipelines = ["HI", "ESI", "T_a"]
    fig, ax = plt.subplots(figsize=(8, 5))
    bottoms = np.zeros(len(pipelines))

    for tier in TIER_PLOT_ORDER:
        vals = []
        for p in pipelines:
            match = tier_day_summary[
                (tier_day_summary["Index Pipeline"] == p) &
                (tier_day_summary["Severity Tier (Daily Peak)"] == tier)
            ]
            vals.append(int(match["Days"].values[0]) if len(match) else 0)
        bars = ax.bar(pipelines, vals, bottom=bottoms, color=TIER_COLOURS[tier],
                      label=tier, edgecolor="white", linewidth=0.8)
        for bar, val, bottom in zip(bars, vals, bottoms):
            if val > 0:
                ax.text(bar.get_x() + bar.get_width() / 2, bottom + val / 2,
                        str(val), ha="center", va="center", fontsize=9, fontweight="bold")
        bottoms += np.array(vals)

    ax.set_ylabel("Days (daily peak severity tier)", fontsize=11)
    ax.set_title("Daily Peak Severity Tier by Index Pipeline\nMohali District, 2022 (UPSDMA framework)",
                 fontsize=12, fontweight="bold")
    ax.legend(title="Severity Tier", fontsize=9)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    plt.tight_layout()
    path = out_dir / "fig1_severity_tier_comparison.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {path.name}")

def plot_index_time_series_tiers(df: pd.DataFrame, tier_thresholds: dict, out_dir: Path):
    fig, axes = plt.subplots(3, 1, figsize=(13, 11), sharex=True)
    pairs = [
        ("HI_max",  "HI",  "Heat Index (HI) [C]"),
        ("ESI_max", "ESI", "Environmental Stress Index (ESI) [C]"),
        ("Ta_max",  "T_a", "Raw Air Temperature T_a [C]"),
    ]
    daily_max = df.resample("D").max(numeric_only=True)

    for ax, (val_col, key, ylabel) in zip(axes, pairs):
        tiers = tier_thresholds[key]
        ax.plot(daily_max.index, daily_max[val_col], color="#1f1f1f",
                linewidth=1.0, label="Daily Max")

        for tier_name in ["Yellow", "Orange", "Red"]:
            ax.axhline(tiers[tier_name], color=TIER_COLOURS[tier_name], linewidth=1.2,
                       linestyle="--", label=f"{tier_name} ({tiers[tier_name]:.1f} C)")

        daily_peak = daily_peak_severity(df, key)
        for date, tier in daily_peak.items():
            if tier != "None":
                ax.axvspan(date, date + pd.Timedelta(days=1), alpha=0.30,
                           color=TIER_COLOURS[tier], linewidth=0)

        ax.set_ylabel(ylabel, fontsize=10)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.legend(fontsize=8, loc="upper right", ncol=2)
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
        ax.xaxis.set_major_locator(mdates.MonthLocator())

    axes[-1].set_xlabel("Month (2022)", fontsize=11)
    fig.suptitle("Daily Maximum Index Values vs UPSDMA-Derived Yellow/Orange/Red Thresholds\n"
                 "Shaded regions show daily peak severity tier -- Mohali District, 2022",
                 fontsize=12, fontweight="bold")
    plt.tight_layout()
    path = out_dir / "fig2_index_time_series_tiers.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {path.name}")

def plot_dual_mode_timeline(mode_daily: pd.Series, forecast: pd.Series, out_dir: Path):
    fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True)
    mode_colours = {"low_power": "#2ca02c", "high_alert": "#d62728"}

    ax = axes[0]
    ax.plot(forecast.index, forecast.values, color="#2c7bb6", linewidth=1.2,
            label="7-Day Rolling Departure (C)")
    ax.axhline(FORECAST_DEPARTURE_THRESHOLD, color="#d62728", linewidth=1.2,
               linestyle="--", label=f"Trigger Threshold ({FORECAST_DEPARTURE_THRESHOLD} C)")
    ax.fill_between(forecast.index, forecast.values,
                    FORECAST_DEPARTURE_THRESHOLD,
                    where=forecast.values >= FORECAST_DEPARTURE_THRESHOLD,
                    alpha=0.3, color="#d62728", label="Exceeds Threshold")
    ax.set_ylabel("T_a Departure from Normal (C)", fontsize=10)
    ax.set_title("Forecast Departure and Dual-Mode Trigger", fontsize=11, fontweight="bold")
    ax.legend(fontsize=9, loc="upper right")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax = axes[1]
    for date, mode in mode_daily.items():
        ax.bar(date, 1, width=1, color=mode_colours[mode], alpha=0.85, linewidth=0)

    from matplotlib.patches import Patch
    legend_elements = [
        Patch(color=mode_colours["low_power"],  label="Low-Power Mode (hourly)"),
        Patch(color=mode_colours["high_alert"], label="High-Alert Mode (15-min)"),
    ]
    ax.legend(handles=legend_elements, fontsize=9, loc="upper right")
    ax.set_ylabel("Sensing Mode", fontsize=10)
    ax.set_yticks([])
    ax.set_title("Daily Sensing Mode Assignment", fontsize=11, fontweight="bold")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    ax.set_xlabel("Month (2022)", fontsize=11)

    plt.tight_layout()
    path = out_dir / "fig3_dual_mode_timeline.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {path.name}")

def plot_delivery_results(mc_df: pd.DataFrame, out_dir: Path):
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    channels  = ["App Push", "News Feed", "Radio Feed", "SMS"]
    net_conds = list(NETWORK_CONDITIONS.keys())
    x = np.arange(len(channels))
    width = 0.35
    palette = ["#2c7bb6", "#d7191c"]

    ax = axes[0]
    for j, net in enumerate(net_conds):
        subset = mc_df[mc_df["Network Condition"] == net]
        rates  = [subset[subset["Channel"] == ch]["Delivery Rate (%)"].values[0]
                  for ch in channels]
        bars = ax.bar(x + j * width, rates, width, label=net, color=palette[j],
                      alpha=0.85, edgecolor="white")
        for bar, val in zip(bars, rates):
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 0.4,
                    f"{val:.0f}%", ha="center", va="bottom", fontsize=8)

    ax.set_xticks(x + width / 2)
    ax.set_xticklabels(channels, fontsize=10)
    ax.set_ylabel("Delivery Success Rate (%)", fontsize=11)
    ax.set_ylim(0, 115)
    ax.set_title("Alert Delivery Rate by Channel and Network Condition", fontsize=11, fontweight="bold")
    ax.legend(fontsize=9)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax = axes[1]
    for j, net in enumerate(net_conds):
        subset = mc_df[mc_df["Network Condition"] == net]
        lats   = []
        for ch in channels:
            val = subset[subset["Channel"] == ch]["Mean Latency (s)"].values[0]
            lats.append(float(val) if val != "N/A" else 0.0)
        bars = ax.bar(x + j * width, lats, width, label=net, color=palette[j],
                      alpha=0.85, edgecolor="white")
        for bar, val in zip(bars, lats):
            if val > 0:
                ax.text(bar.get_x() + bar.get_width() / 2,
                        bar.get_height() + 0.5,
                        f"{val:.0f}s", ha="center", va="bottom", fontsize=8)

    ax.set_xticks(x + width / 2)
    ax.set_xticklabels(channels, fontsize=10)
    ax.set_ylabel("Mean Delivery Latency (s)", fontsize=11)
    ax.set_title("Mean Alert Delivery Latency by Channel and Network Condition", fontsize=11, fontweight="bold")
    ax.legend(fontsize=9)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    fig.suptitle(f"Monte Carlo Delivery Simulation ({MC_ITERATIONS:,} iterations) -- Mohali District 2022",
                 fontsize=12, fontweight="bold", y=1.02)
    plt.tight_layout()
    path = out_dir / "fig4_delivery_simulation.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {path.name}")


## 12. Run the full pipeline

In [16]:
def print_table(title: str, df_or_dict):
    width = 72
    print("\n" + "=" * width)
    print(f"  {title}")
    print("=" * width)
    if isinstance(df_or_dict, dict):
        for k, v in df_or_dict.items():
            print(f"  {k:<42} {v}")
    else:
        print(df_or_dict.to_string(index=False))
    print("=" * width)

def run(grib_path, baseline_jan_jun_path, baseline_jul_dec_path, out_dir: Path):
    out_dir.mkdir(parents=True, exist_ok=True)
    print("\n[1/7] Loading ERA5 data...")

    if grib_path and Path(grib_path).exists():
        df_raw = load_era5_grib(grib_path)
    else:
        print("  No GRIB file found -- using synthetic ERA5-like data.")
        df_raw = generate_synthetic_era5()

    print("\n[2/7] Computing heat stress indices...")
    df_idx = compute_indices(df_raw)

    print("\n[3/7] Simulating 25 sensor nodes with UHI offsets...")
    df_nodes = simulate_nodes(df_idx)

    print("\n[4/7] Deriving Yellow/Orange/Red tier thresholds...")
    have_baseline = (baseline_jan_jun_path and baseline_jul_dec_path
                      and Path(baseline_jan_jun_path).exists()
                      and Path(baseline_jul_dec_path).exists())
    if have_baseline:
        df_jan_jun = load_era5_grib(baseline_jan_jun_path, apply_sim_slice=False)
        df_jul_dec = load_era5_grib(baseline_jul_dec_path, apply_sim_slice=False)
        baseline_raw = pd.concat([df_jan_jun, df_jul_dec]).sort_index()
        baseline_idx = compute_indices(baseline_raw)
        tier_thresholds = compute_baseline_tier_thresholds(baseline_idx)
        clim_normal = compute_climatological_normal(baseline_idx)
        print("  T_a: UPSDMA published values (Ludhiana proxy for Mohali)")
        print("  HI/ESI: derived from real ERA5 1991-2020 baseline (11:30-17:30 window)")
    else:
        tier_thresholds = compute_baseline_tier_thresholds()
        clim_normal = None
        print("  T_a: UPSDMA published values (Ludhiana proxy for Mohali)")
        print("  HI/ESI: derived from placeholder baseline -- provide")
        print("  baseline_jan_jun_path/baseline_jul_dec_path for real ERA5 data.")
    for idx_name, tiers in tier_thresholds.items():
        print(f"    {idx_name:>4}:  Yellow={tiers['Yellow']:.2f}   "
              f"Orange={tiers['Orange']:.2f}   Red={tiers['Red']:.2f}")

    print("\n[5/7] Classifying severity tiers...")
    df_alerts = detect_severity_tiers(df_nodes, tier_thresholds)
    df_alerts_heat_season = df_alerts.loc[HEAT_SEASON_START:HEAT_SEASON_END]

    print("\n[6/7] Simulating dual-mode sensor operation...")
    df_mode, mode_daily, forecast = simulate_dual_mode(df_alerts, clim_mean_series=clim_normal)

    print("\n[7/7] Running Monte Carlo delivery simulation...")
    mc_df = monte_carlo_delivery()

    tier_day_summary = summarise_severity_tiers(df_alerts_heat_season)
    print_table("RESULT 1 -- Daily Peak Severity Tier Counts by Index Pipeline", tier_day_summary)

    tier_hour_summary = summarise_severity_hours(df_alerts_heat_season)
    print_table("RESULT 1b -- Hourly Severity Tier Counts by Index Pipeline", tier_hour_summary)

    dual_stats = summarise_dual_mode(mode_daily, forecast)
    print_table("RESULT 2 -- Dual-Mode Sensor Operation Statistics", dual_stats)

    df_mode_alert = df_alerts.copy()
    df_mode_alert["mode"] = df_mode["mode"]
    mode_tier_summary = summarise_severity_by_mode(df_mode_alert)
    print_table("RESULT 3 -- Severity Tier Hours by Sensing Mode", mode_tier_summary)

    print_table("RESULT 4 -- Monte Carlo Alert Delivery Simulation",
                mc_df.sort_values(["Channel", "Network Condition"]))

    tier_day_summary.to_csv(out_dir / "result1_severity_tiers_daily.csv", index=False)
    tier_hour_summary.to_csv(out_dir / "result1b_severity_tiers_hourly.csv", index=False)
    pd.DataFrame([dual_stats]).to_csv(out_dir / "result2_dual_mode.csv", index=False)
    mode_tier_summary.to_csv(out_dir / "result3_severity_by_mode.csv", index=False)
    mc_df.to_csv(out_dir / "result4_delivery.csv", index=False)
    print(f"\n  CSV results saved to: {out_dir}")

    print("\n  Generating figures...")
    plot_severity_tier_comparison(tier_day_summary, out_dir)
    plot_index_time_series_tiers(df_alerts, tier_thresholds, out_dir)
    plot_dual_mode_timeline(mode_daily, forecast, out_dir)
    plot_delivery_results(mc_df, out_dir)

    print(f"\n  All outputs written to: {out_dir}\n")
    return tier_day_summary, tier_hour_summary, dual_stats, mode_tier_summary, mc_df, df_alerts, tier_thresholds

(tier_day_summary, tier_hour_summary, dual_stats,
 mode_tier_summary, mc_df, df_alerts, tier_thresholds) = run(
    grib_path=GRIB_PATH,
    baseline_jan_jun_path=BASELINE_JAN_JUN_PATH,
    baseline_jul_dec_path=BASELINE_JUL_DEC_PATH,
    out_dir=OUT_DIR,
)



[1/7] Loading ERA5 data...
  Loading GRIB: Mohali2022New.grib
  Variables in file (primary load): ['d2m', 't2m']
  Loaded T_a from 't2m'
  Loaded Td from 'd2m'
  Could not load SSRD via filter_by_keys: axis 1 is out of bounds for array of dimension 1
  Targeted load for paramId 169 did not yield 'ssrd'.
  Loaded 8,760 records (2022-01-01 00:00:00 to 2022-12-31 23:00:00)

[2/7] Computing heat stress indices...

[3/7] Simulating 25 sensor nodes with UHI offsets...

[4/7] Deriving Yellow/Orange/Red tier thresholds...
  Loading GRIB: 19991_to_2020.grib
  Variables in file (primary load): ['d2m', 't2m']
  Loaded T_a from 't2m'
  Loaded Td from 'd2m'
  Could not load SSRD via filter_by_keys: axis 1 is out of bounds for array of dimension 1
  Targeted load for paramId 169 did not yield 'ssrd'.
  Loaded 38,066 records (1991-01-01 06:00:00 to 2020-06-30 12:00:00)
  Loading GRIB: 19991_to_2020_two.grib
  Variables in file (primary load): ['d2m', 't2m']
  Loaded T_a from 't2m'
  Loaded Td from '

NameError: name 'compute_baseline_tier_thresholds' is not defined

## 13. (Optional) Download results as a zip

In [ ]:
import zipfile
import os
from google.colab import files as colab_files

GRIB_PATH = '/content/Mohali2022New.grib'

print("Starting EDA for meteorological data...")

if GRIB_PATH and Path(GRIB_PATH).exists():
    df_eda = load_era5_grib(GRIB_PATH)
else:
    print("No GRIB file found (or path invalid) -- using synthetic ERA5-like data for EDA.")
    df_eda = generate_synthetic_era5()

df_eda = compute_indices(df_eda);

print("\n--- Data Head ---")
print(df_eda.head())

print("\n--- Data Info ---")
df_eda.info()

print("\n--- Data Description ---")
print(df_eda.describe())

print("\nGenerating initial plots...")
fig, axes = plt.subplots(nrows=6, ncols=1, figsize=(15, 18), sharex=True)

df_eda['T_a'].plot(ax=axes[0], title='Air Temperature (T_a)', ylabel='°C', color='red')
df_eda['Td'].plot(ax=axes[1], title='Dew Point Temperature (Td)', ylabel='°C', color='blue')
df_eda['RH'].plot(ax=axes[2], title='Relative Humidity (RH)', ylabel='%', color='green')
df_eda['HI'].plot(ax=axes[3], title='Heat Index (HI)', aylabel='°C', color='orange')
df_eda['ESI'].plot(ax=axes[4], title='Environmental Stress Index (ESI)', ylabel='°C', color='purple')
df_eda['SSRD'].plot(ax=axes[5], title='Surface Solar Radiation Downwards (SSRD)', ylabel='W/m²', color='gray')

for ax in axes:
    ax.grid(True, linestyle='--', alpha=0.6)
    ax.set_xlabel('')

plt.xlabel('Time')
plt.tight_layout()
plt.show()
print("EDA complete. You can now explore the plots and statistics above.")


In [ ]:
import xarray as xr

grib_path = GRIB_PATH

ds_check = xr.open_dataset(
    grib_path, engine="cfgrib",
    backend_kwargs={"filter_by_keys": {"paramId": 169}}
)
print(list(ds_check.data_vars))


In [ ]:
import shutil
from google.colab import files as colab_files

zip_path = shutil.make_archive("/content/heat_stress_results", "zip", OUT_DIR)
colab_files.download(zip_path)
